<a target="_blank" href="https://colab.research.google.com/github/mim-ml-teaching/public-nlp-2025-26/blob/main/hws/NLP_HW1_Speculative_Decoding.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>

# NLP Homework 1: Speculative Decoding

## Background

In standard autoregressive transformers, tokens are generated strictly one at a time. Each new token requires a full forward pass, and because generation is causal, the model cannot produce the $k^\text{th}$ token until all preceding tokens have been processed.

This bottleneck is addressed by *Speculative decoding*. A smaller, faster "draft" model first proposes a sequence of candidate tokens. The larger target model then evaluates it in a single forward pass. If the proposed tokens are in fact assigned high-probabilities by a target model then they are accepted, saving additional computation. If some of the proposed tokens are incorrect, the sequence of tokens is rejected at the point of divergence. Although this requires partial recomputation, the larger model has already processed the prefix in parallel.

**Core idea** Given a prefix, a cheap *draft model* proposes $K$ subsequent tokens: $t_1, \dots, t_K$.
The expensive *target model* is then run **once** on the extended sequence: $[\text{prefix}, t_1, \dots, t_K]$.
This single forward pass produces logits for all $K+1$ positions in parallel.

**Verification**  The target model ($p_{\text{target}}(\cdot \mid \text{prefix})$) is used to verify the proposed tokens. That is,
for $i = 1, \dots, K$, we check whether
$$
t_i = \arg\max\; p_{\text{target}}(\cdot \mid \text{prefix}, t_1, \dots, t_{i-1}).
$$
This can be done in parallel! Now, let $j$ be the largest index such that for all $t_1, \dots, t_j$ the above equality holds. It turns out, we can accept $t_1, \dots, t_j$, discard the rest, and also take the target model's prediction at position $j+1$.

## Tasks

You will work with WikiText-2 (`wikitext-2-raw-v1`) dataset, and target (verifier) model `google/gemma-3-4b-pt`.


### Task 1: Implement Speculative Decoding Loop (4 pts)

Implement two functions:

1. **`verify_draft_tensor(context_ids, draft_ids, model)`** - Given a context and a sequence of $K$ draft tokens, run a single forward pass of the target model on the concatenated sequence and determine how many draft tokens match the target model's greedy predictions. Return the accepted prefix plus the target model's next-token prediction. **Your implementation must be fully vectorized (no `for` loops)** to receive maximum number of points.

2. **`speculative_generate_tensor(context_ids, draft_fn, model, n_tokens, k)`** - The outer generation loop that repeatedly calls the draft function and verifier until `n_tokens` new tokens have been generated. Track and return statistics (e.g., total draft tokens proposed, total accepted, number of verification rounds). Try to avoid device to host transfers in the main loop.

See the code cell below for the exact function signatures.

### Task 2: Design a Draft Model & Evaluate (4 pts)

Design and implement a **transformer-based** draft model that proposes $K$ candidate tokens given a context. You may either:

- Use a pre-trained small model (e.g., `google/gemma-3-1b-pt`), or
- Train a small transformer from scratch (e.g., via distillation from the target model, or training directly on `wikitext-2-raw-v1`).

**Deliverables:**

1. Implement your draft model and wrap it so it conforms to the `draft_fn(context_ids, k)` interface (see `speculative_generate_tensor` argument, or `bigram_draft` function).
2. Evaluate on the WikiText-2 **test** split (`test_ds`).
3. Report the acceptance rate (fraction of draft tokens accepted by the target model).
4. Compare against the provided bigram baseline - your model should achieve a higher acceptance rate.
5. Measure wall time of speculative decoding vs standard execution. Your solution should achieve a speedup.

### Task 3: Visualization & Report (2 pts)

Write a concise summary (≤ 300 words) describing your approach, key findings, and observations.

Your summary should include at least two informative plots. For example you may visualize:

* Speedup vs model size (if you train your own transformer)
* Acceptance rate vs context length for draft model
* Histogram of accepted draft lengths

All plots must have axis labels, legends, and titles.

In [1]:
import random
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict

random.seed(42)
torch.manual_seed(42)

TARGET_MODEL = "google/gemma-3-4b-pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if device == "cuda" else torch.float32
print(f"Running on: {device}")

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

Running on: cuda


In [7]:
from huggingface_hub import login
login()

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
target_model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, torch_dtype=dtype).to(device)
target_model.eval()

n_params_target = sum(p.numel() for p in target_model.parameters()) / 1e6
print(f"Target model: {TARGET_MODEL}  ({n_params_target:.0f}M params)  on {device}")

config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Target model: google/gemma-3-4b-pt  (4300M params)  on cuda


In [9]:
def autoregressive_generate(context_ids: list[int], n_tokens: int) -> list[int]:
    """Standard autoregressive generation with the target model (baseline)."""
    input_ids = torch.tensor([context_ids], device=device)
    for _ in range(n_tokens):
        logits = target_model(input_ids).logits
        next_tok = logits[:, -1:].argmax(dim=-1)
        input_ids = torch.cat([input_ids, next_tok], dim=1)
    return input_ids[0, len(context_ids):].tolist()

In [10]:
def get_split(split):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
    examples = [row["text"] for row in ds if row["text"].strip()]
    tokenized_examples = [tokenizer.encode(text, add_special_tokens=False) for text in examples]
    num_tokens = sum(len(tokens) for tokens in tokenized_examples)
    print(f"{split.capitalize()} split: {len(examples):,} examples, {num_tokens:,} tokens in total.")
    return tokenized_examples

train_ds = get_split("train")
test_ds = get_split("test")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train split: 23,767 examples, 2,459,301 tokens in total.
Test split: 2,891 examples, 292,281 tokens in total.


# Naive draft model (bigrams trained on the dataset)


In [11]:
train_ids = [tok for example in train_ds for tok in example]
bigram_counts = defaultdict(lambda: defaultdict(int))
for id1, id2 in zip(train_ids, train_ids[1:]):
    bigram_counts[id1][id2] += 1

bigram_next = {}
for tok, successors in bigram_counts.items():
    bigram_next[tok] = max(successors, key=successors.get)

fallback_token = tokenizer.encode("the", add_special_tokens=False)[0]

def bigram_draft(context_ids: list[int], k: int) -> list[int]:
    draft = []
    last_tok = context_ids[-1]
    for _ in range(k):
        next_tok = bigram_next.get(last_tok, fallback_token)
        draft.append(next_tok)
        last_tok = next_tok
    return draft

In [12]:
# Example run

prompt = "The history of artificial"
ctx = tokenizer.encode(prompt, add_special_tokens=False)

draft = bigram_draft(ctx, k=3)
print(f"Context: {prompt!r}")
print(f"Bigram draft: {tokenizer.decode(draft)!r}")

Context: 'The history of artificial'
Bigram draft: ' intelligence . '


# Task 1: Implement speculative decoding

In [14]:
@torch.no_grad()
def verify_draft_tensor(
    context_ids: torch.Tensor,
    draft_ids: torch.Tensor,
    model,
) -> torch.Tensor:
    """
    Verify draft tokens against a model.

    Args:
        context_ids: 1-D tensor (n,) — context tokens.
        draft_ids:   1-D tensor (k,) — drafted token ids to verify.
        model:       the verifier model.

    Returns:
        1-D tensor — verified context + accepted predictions + next token.
    """

    all_ids = torch.cat([context_ids,draft_ids])
    logits = model(all_ids.unsqueeze(0)).logits
    logits = logits[0]
    target_preds = logits.argmax(dim=-1)

    n = len(context_ids)
    k = len(draft_ids)
    draft_preds = target_preds[n-1:n+k-1]

    matches = (draft_preds == draft_ids)
    n_matches = matches.int().cumprod(dim=-1).sum()

    return torch.cat([context_ids,draft_ids[:n_matches],target_preds[n-1+n_matches].unsqueeze(0)])

@torch.no_grad()
def speculative_generate_tensor(
    context_ids: list[int],
    draft_fn,
    model,
    n_tokens: int,
    k: int,
) -> tuple[list[int], dict]:
    """
    Speculative generation.

    Args:
        context_ids: prefix token ids.
        draft_fn:    callable(sequence_tensor, k) -> (k,) device tensor.
        model:       verifier model.
        n_tokens:    number of tokens to generate.
        k:           speculation length.

    Returns:
        (generated_token_ids, stats_dict)
    """
    context_ids = torch.tensor(context_ids, device=device)
    start_len = len(context_ids)
    while len(context_ids) < start_len + n_tokens :
        draft_ids = draft_fn(context_ids,k)
        context_ids = verify_draft_tensor(context_ids,draft_ids,model)

    return context_ids.tolist(), {}

# Task 2: Evaluate speculative decoding


In [ ]:
# TODO


# Task 3: Visualization & Report


In [ ]:
# TODO: Your plots

*TODO: Write your report here.*